In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"
if [ "$(uname -sm)" != "Linux x86_64" ]; then
    echo " requires x86-64 Linux." >&2
    exit 1
fi
mkdir -p "$work"

conda="$(command -v conda || printf '%s' "$work/miniforge/bin/conda")"
if [ ! -x "$conda" ]; then
    curl -fLo "$work/miniforge.sh" https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    bash "$work/miniforge.sh" -b -p "$work/miniforge"
fi

repo="$work/EMProt"
if [ ! -d "$repo" ]; then
    git clone https://github.com/huang-laboratory/EMProt.git "$repo"
fi
git -C "$repo" checkout --quiet --detach 9489d52

env="$work/EMProt-env"
CONDA_CHANNEL_PRIORITY=flexible "$conda" env update -p "$env" -f "$repo/environment.yml"

cd "$repo"
"$conda" run -p "$env" python -m pip install -e .
if [ ! -d emprot/weights/model_all_atom ]; then
    curl -fL http://huanglab.phys.hust.edu.cn/EMProt/pretrained_weights/weights.tgz | tar -xz -C emprot
fi


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  122M  100  122M    0     0  51.5M      0  0:00:02  0:00:02 --:--:-- 68.5M


PREFIX=/workspace/mica-benchmark/archives/cryo-benchmark/miniforge
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.10.23-hbd8a1cb_0.conda
Extracting libgomp-16.2.0-he0feb66_7.conda
Extracting libzlib-1.3.2-h25fd6f3_3.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_2.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.14-9_cp314.conda
Extracting tzdata-2026e-h5d301ff_0.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_7.conda
Extracting ld_impl_linux-64-2.46.1-default_hbd61a6d_103.conda
Extracting libgcc-16.2.0-ha9f2e26_7.conda
Extracting bzip2-1.0.8-hda65f42_10.conda
Extracting c-ares-1.34.9-hebe6cf0_0.conda
Extracting keyutils-1.6.3-h7cc23a3_1.conda
Extracting libev-4.33-h280c20c_3.conda
Extracting libexpat-2.9.0-hd2095e1_0.conda
Extracting libffi-3.7.0-h81df57d_1.conda
Extracting libiconv-1.18-h0cb94f2_3.conda
Extracting liblzma-5.8.3-hb03c661_1.conda
Extracting libmpdec-4.0.0-hb03c661_2.conda
Extrac

warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.


Linking ca-certificates-2026.10.23-hbd8a1cb_0
Linking libgomp-16.2.0-he0feb66_7
Linking libzlib-1.3.2-h25fd6f3_3
Linking nlohmann_json-abi-3.12.0-h0f90c79_2
Linking pybind11-abi-11-hc364b38_1
Linking python_abi-3.14-9_cp314
Linking tzdata-2026e-h5d301ff_0
Linking _openmp_mutex-4.5-20_gnu
Linking zstd-1.5.7-hb78ec9c_7
Linking ld_impl_linux-64-2.46.1-default_hbd61a6d_103
Linking libgcc-16.2.0-ha9f2e26_7
Linking bzip2-1.0.8-hda65f42_10
Linking c-ares-1.34.9-hebe6cf0_0
Linking keyutils-1.6.3-h7cc23a3_1
Linking libev-4.33-h280c20c_3
Linking libexpat-2.9.0-hd2095e1_0
Linking libffi-3.7.0-h81df57d_1
Linking libiconv-1.18-h0cb94f2_3
Linking liblzma-5.8.3-hb03c661_1
Linking libmpdec-4.0.0-hb03c661_2
Linking libpython-3.14.10-hd51c5da_100_cp314
Linking libstdcxx-16.2.0-h934c35e_7
Linking libuuid-2.42.4-hcfc3c73_1
Linking lzo-2.10-hebe6cf0_1003
Linking ncurses-6.6-hdb14827_1
Linking openssl-3.6.5-h781a0a9_0
Linking reproc-14.2.8.post0-hee9eb32_0
Linking tk-8.6.13-noxft_h1df4ec4_4
Linking cpp-expe

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  164M  100  164M    0     0  1121k      0  0:02:30  0:02:30 --:--:-- 3981k


In [3]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

conda="$(command -v conda || printf '%s' "$work/miniforge/bin/conda")"

method=EMProt
export MPLBACKEND=Agg
cd "$work/EMProt"
env="$work/EMProt-env"

for target in 22XP 32MC; do
    data="$root/data/targets/$target"
    map=("$data"/inputs/emd_*.map)
    fasta="$data/inputs/rcsb_pdb_${target}.fasta"
    out="$work/runs/$method/$target"
    mkdir -p "$out"

    cif=("$data"/predictions/af3/*/*_model_0.cif)
    "$conda" run -p "$env" --no-capture-output python - "${cif[0]}" "$out/af3_chain.pdb" <<'PYCODE'
import sys
from Bio.PDB import MMCIFParser, PDBIO

structure = MMCIFParser(QUIET=True).get_structure("af3", sys.argv[1])
writer = PDBIO()
writer.set_structure(next(structure.get_chains()))
writer.save(sys.argv[2])
PYCODE
    time "$conda" run -p "$env" --no-capture-output emprot build --map "${map[0]}" \
        --seq "$fasta" --chain "$out/af3_chain.pdb" --device 0 --output "$out"
done


# Job start at 2026-11-23 07:00:55.846354
# Output dir set to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/22XP
# Temp   dir set to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/22XP/temp
Write formatted seq to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/22XP/temp/format_seq.fasta
User input 1 single-chain structure templates
Using single-chain structure as templates
Rewrite template 0 from single-chain to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/22XP/temp/chain_templs/templ_0.pdb
# Has chain   template = True
# has complex template = False
# No complex template, use chain from single as template
# Rescale voxel size from [0.96 0.96 0.96] to [1. 1. 1.], shift origin by None
# Write formated map to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/22XP/temp/format_map.mrc
# Time consupmtion 6.0048 seconds. Done preprocess
# Script dir is /workspace/mica-benchmark/archives/cryo-benchmark/EMProt/emprot/pip


real	10m15.431s
user	9m29.232s
sys	0m41.685s


# Job start at 2026-11-23 07:11:17.653018
# Output dir set to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/32MC
# Temp   dir set to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/32MC/temp
Write formatted seq to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/32MC/temp/format_seq.fasta
User input 1 single-chain structure templates
Using single-chain structure as templates
Rewrite template 0 from single-chain to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/32MC/temp/chain_templs/templ_0.pdb
# Has chain   template = True
# has complex template = False
# No complex template, use chain from single as template
# Rescale voxel size from [0.82 0.82 0.82] to [1. 1. 1.], shift origin by None
# Write formated map to /workspace/mica-benchmark/archives/cryo-benchmark/runs/EMProt/32MC/temp/format_map.mrc
# Time consupmtion 7.5032 seconds. Done preprocess
# Script dir is /workspace/mica-benchmark/archives/cryo-benchmark/EMProt/emprot/pip


real	10m0.464s
user	11m32.487s
sys	0m27.671s


In [4]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

if [ "$(uname -s)" = Linux ] && ! command -v g++ >/dev/null; then
    apt-get update
    apt-get install -y g++
fi

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

method="EMProt"
if [ ! -d "$work/USalign" ]; then
    git clone https://github.com/pylelab/USalign.git "$work/USalign"
fi
make -C "$work/USalign" USalign

for target in 22XP 32MC; do
    out="$work/runs/$method/$target"
    reference="$root/data/targets/$target/reference/$target.cif"
    model="$out/output.cif"
    test -s "$model"
    "$work/USalign/USalign" "$model" "$reference" -mol prot -mm 1 -ter 1 -outfmt 2 > "$out/usalign.tsv"
    if [ -f "$phenix_env" ]; then
        (source "$phenix_env"; phenix.chain_comparison "$reference" "$model" \
            max_dist=3 unique_part_of_target_only=False test_unique_part_of_target_only=False \
            use_crystal_symmetry=False) > "$out/chain-comparison.txt"
    fi
done


make: Entering directory '/workspace/mica-benchmark/archives/cryo-benchmark/USalign'
make: 'USalign' is up to date.
make: Leaving directory '/workspace/mica-benchmark/archives/cryo-benchmark/USalign'
